# Lenovo Desktop Cleanup & File Management
**Ronald's Desktop — C:\\Users\\equat\\Desktop**

Run cells top to bottom. Each section audits one problem area.  
**DRY_RUN = True** by default — nothing moves until you flip it to False.

In [ ]:
import os
import shutil
import json
from pathlib import Path
from datetime import datetime, timedelta
from collections import defaultdict

DESKTOP = Path(r'C:\Users\equat\Desktop')
DRY_RUN = True  # <-- flip to False when ready to execute

ARCHIVE_ROOT = DESKTOP / 'MUSIC PROJECT FOLDERS' / '_ARCHIVED'
LOG_PATH = Path(r'E:\SunoMaster\scripts\desktop_cleanup_log.txt')

log_lines = []

def log(msg):
    print(msg)
    log_lines.append(msg)

def move(src, dst, reason=''):
    src, dst = Path(src), Path(dst)
    tag = '[DRY RUN]' if DRY_RUN else '[MOVED]'
    log(f'{tag} {src.name}  ->  {dst}  ({reason})')
    if not DRY_RUN:
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(str(src), str(dst))

def delete(path, reason=''):
    path = Path(path)
    tag = '[DRY RUN]' if DRY_RUN else '[DELETED]'
    log(f'{tag} DELETE {path}  ({reason})')
    if not DRY_RUN:
        if path.is_dir():
            shutil.rmtree(path)
        else:
            path.unlink()

print(f'Desktop: {DESKTOP}')
print(f'DRY_RUN = {DRY_RUN}')
print(f'Date: {datetime.now().strftime("%Y-%m-%d %H:%M")}')


## 1. Live Desktop Scan
Full snapshot of current state — compare against last documented structure.

In [ ]:
# Full recursive scan
def scan_desktop(root, depth=0, max_depth=3):
    entries = {'folders': [], 'files': [], 'total_size_mb': 0}
    try:
        for item in sorted(root.iterdir()):
            if item.is_dir():
                entries['folders'].append(item)
            else:
                entries['files'].append(item)
                try:
                    entries['total_size_mb'] += item.stat().st_size / (1024*1024)
                except:
                    pass
    except PermissionError:
        pass
    return entries

top = scan_desktop(DESKTOP)
print(f'TOP-LEVEL FOLDERS ({len(top["folders"])}):')
for f in top['folders']:
    sub = list(f.iterdir()) if f.exists() else []
    sub_dirs = [x for x in sub if x.is_dir()]
    sub_files = [x for x in sub if x.is_file()]
    print(f'  {f.name:<35} [{len(sub_dirs)} folders, {len(sub_files)} files]')

print(f'\nLOOSE FILES ON DESKTOP ROOT ({len(top["files"])}):')
for f in top['files']:
    try:
        size = f.stat().st_size / 1024
        mtime = datetime.fromtimestamp(f.stat().st_mtime).strftime('%Y-%m-%d')
    except:
        size, mtime = 0, 'unknown'
    print(f'  {f.name:<50} {size:>8.1f} KB  {mtime}')


## 2. Audit — MISCEL Duplication
MISCEL has: Diverse, DIVERSEE, Odds, ODDSS — clearly duplicated names. Flatten and consolidate.

In [ ]:
miscel = DESKTOP / 'MISCEL'
print('=== MISCEL CONTENTS ===')
if miscel.exists():
    for item in sorted(miscel.rglob('*')):
        rel = item.relative_to(miscel)
        depth = len(rel.parts)
        indent = '  ' * (depth - 1)
        try:
            size = f'{item.stat().st_size/1024:.1f} KB' if item.is_file() else ''
        except:
            size = ''
        kind = 'FILE' if item.is_file() else 'DIR'
        print(f'{indent}[{kind}] {item.name}  {size}')
else:
    print('MISCEL folder not found')

# Check for near-duplicate subfolder names
print('\n=== DUPLICATE NAME SUSPECTS ===')
all_folders = [f.name.upper() for f in top['folders']]
miscel_subs = [f.name for f in miscel.iterdir() if f.is_dir()] if miscel.exists() else []
print('MISCEL subfolders:', miscel_subs)

# Flag duplicates: Diverse/DIVERSEE, Odds/ODDSS
dup_groups = [
    [miscel / 'Diverse', miscel / 'DIVERSEE'],
    [miscel / 'Odds', miscel / 'ODDSS'],
]
for group in dup_groups:
    existing = [p for p in group if p.exists()]
    if len(existing) > 1:
        print(f'\nDUPLICATE GROUP: {[p.name for p in existing]}')
        for p in existing:
            items = list(p.rglob('*'))
            files = [x for x in items if x.is_file()]
            print(f'  {p.name}: {len(files)} files')


In [ ]:
# ACTION: Merge DIVERSEE into Diverse, ODDSS into Odds, then delete empties
# Review the audit above first — then run this cell

def merge_folders(src_folder, dst_folder):
    src, dst = Path(src_folder), Path(dst_folder)
    if not src.exists():
        log(f'SKIP: {src.name} does not exist')
        return
    for item in src.rglob('*'):
        if item.is_file():
            rel = item.relative_to(src)
            target = dst / rel
            if not target.exists():
                move(item, target, f'merge {src.name} -> {dst.name}')
            else:
                log(f'[SKIP-EXISTS] {item.name} already in {dst.name}')
    # Remove src if empty after merge
    if not DRY_RUN:
        remaining = list(src.rglob('*'))
        if not remaining:
            delete(src, 'empty after merge')

merge_folders(miscel / 'DIVERSEE', miscel / 'Diverse')
merge_folders(miscel / 'ODDSS', miscel / 'Odds')


## 3. Audit — OLD DOWNLOADS in Apps
Apps/OLD DOWNLOADS — likely installer files no longer needed.

In [ ]:
old_dl = DESKTOP / 'Apps' / 'OLD DOWNLOADS'
print(f'=== OLD DOWNLOADS ===' )
if old_dl.exists():
    items = sorted(old_dl.rglob('*'))
    files = [x for x in items if x.is_file()]
    total_mb = sum(x.stat().st_size for x in files if x.exists()) / (1024*1024)
    print(f'Total: {len(files)} files, {total_mb:.1f} MB')
    print()
    for f in files:
        try:
            size = f.stat().st_size / (1024*1024)
            mtime = datetime.fromtimestamp(f.stat().st_mtime).strftime('%Y-%m-%d')
        except:
            size, mtime = 0, 'unknown'
        print(f'  {f.name:<60} {size:>6.1f} MB  {mtime}')
else:
    print('Folder not found or empty')


In [ ]:
# ACTION: Delete OLD DOWNLOADS entirely (review audit above first)
# Uncomment to execute:
# delete(old_dl, 'stale installer files — reviewed and approved')
print('Uncomment the delete line above after reviewing the contents.')


## 4. Audit — TEMPORARY Folder
Should only contain Screen Saves (used for phone project). Everything else is clutter.

In [ ]:
temp_folder = DESKTOP / 'TEMPORARY'
print('=== TEMPORARY CONTENTS ===')
if temp_folder.exists():
    for item in sorted(temp_folder.rglob('*')):
        rel = item.relative_to(temp_folder)
        try:
            size = f'{item.stat().st_size/1024:.1f} KB' if item.is_file() else ''
            mtime = datetime.fromtimestamp(item.stat().st_mtime).strftime('%Y-%m-%d')
        except:
            size, mtime = '', 'unknown'
        kind = 'FILE' if item.is_file() else 'DIR '
        print(f'  [{kind}] {str(rel):<55} {size:<12} {mtime}')
else:
    print('TEMPORARY folder not found')


## 5. Audit — MUSIC PROJECT FOLDERS
26 active projects — identify which are finished/released and can be archived.

In [ ]:
mpf = DESKTOP / 'MUSIC PROJECT FOLDERS'
print('=== MUSIC PROJECT FOLDERS ===')
print(f'{"Project":<40} {"Subfolders":>10} {"Files":>8} {"Size MB":>10} {"Last Modified"}')
print('-' * 90)

project_data = []
if mpf.exists():
    for proj in sorted(mpf.iterdir()):
        if not proj.is_dir() or proj.name.startswith('_'):
            continue
        all_items = list(proj.rglob('*'))
        files = [x for x in all_items if x.is_file()]
        dirs = [x for x in all_items if x.is_dir()]
        size_mb = sum(x.stat().st_size for x in files if x.exists()) / (1024*1024)
        mtimes = [x.stat().st_mtime for x in files if x.exists()]
        last_mod = datetime.fromtimestamp(max(mtimes)).strftime('%Y-%m-%d') if mtimes else 'empty'
        project_data.append({'name': proj.name, 'path': proj, 'files': len(files),
                              'dirs': len(dirs), 'size_mb': size_mb, 'last_mod': last_mod})
        print(f'{proj.name:<40} {len(dirs):>10} {len(files):>8} {size_mb:>10.1f} {last_mod}')

print(f'\nTotal projects: {len(project_data)}')

# Flag projects not touched in 6+ months
cutoff = datetime.now() - timedelta(days=180)
print('\n--- Projects NOT modified in 6+ months (archive candidates) ---')
for p in project_data:
    try:
        mod = datetime.strptime(p['last_mod'], '%Y-%m-%d')
        if mod < cutoff:
            print(f'  {p["name"]:<40} last: {p["last_mod"]}')
    except:
        pass


In [ ]:
# ACTION: Archive completed projects
# Add project names to this list after reviewing the audit above

ARCHIVE_THESE = [
    # Example: 'FLOWERS',
    # 'Bloomberg',
    # Add names here tomorrow
]

if ARCHIVE_THESE:
    if not DRY_RUN:
        ARCHIVE_ROOT.mkdir(parents=True, exist_ok=True)
    for name in ARCHIVE_THESE:
        src = mpf / name
        dst = ARCHIVE_ROOT / name
        if src.exists():
            move(src, dst, 'archived completed project')
        else:
            log(f'[NOT FOUND] {name}')
else:
    print('ARCHIVE_THESE list is empty — populate it after reviewing the audit above.')


## 6. Audit — Apps Folder (44 subcategories)
Check for empty categories, duplicates, and shortcuts pointing to non-existent targets.

In [ ]:
import subprocess

apps = DESKTOP / 'Apps'
print(f'=== APPS SUBCATEGORIES ({len(list(apps.iterdir()))} total) ===')
print(f'{"Category":<45} {"Shortcuts":>10} {"Empty?"}')
print('-' * 65)

empty_cats = []
lnk_broken = []

for cat in sorted(apps.iterdir()):
    if not cat.is_dir():
        continue
    items = list(cat.rglob('*'))
    lnks = [x for x in items if x.suffix.lower() in ('.lnk', '.url')]
    is_empty = len(items) == 0
    if is_empty:
        empty_cats.append(cat)
    print(f'{cat.name:<45} {len(lnks):>10}  {"EMPTY" if is_empty else ""}')

print(f'\nEMPTY CATEGORIES ({len(empty_cats)}):')
for c in empty_cats:
    print(f'  {c.name}')


In [ ]:
# Check for broken .lnk shortcuts (Windows shell check)
print('=== BROKEN SHORTCUTS CHECK ===')
print('Scanning .lnk files...')

broken = []
for lnk in apps.rglob('*.lnk'):
    # Use PowerShell to resolve shortcut target
    try:
        result = subprocess.run(
            ['powershell', '-Command',
             f'$sh = New-Object -ComObject WScript.Shell; $sc = $sh.CreateShortcut("{lnk}"); $sc.TargetPath'],
            capture_output=True, text=True, timeout=5
        )
        target = result.stdout.strip()
        if target and not Path(target).exists():
            broken.append((lnk, target))
    except Exception as e:
        pass

print(f'Broken shortcuts: {len(broken)}')
for lnk, target in broken:
    rel = lnk.relative_to(apps)
    print(f'  {str(rel):<60} -> {target}')


In [ ]:
# ACTION: Delete empty App categories and broken shortcuts
# Run audit cells above first

# Delete empty categories
for cat in empty_cats:
    delete(cat, 'empty App category')

# Delete broken shortcuts
for lnk, target in broken:
    delete(lnk, f'broken shortcut -> {target}')


## 7. Audit — MUSIC OUTPUT
Check for duplicate songs across: 2025 Songs, Latest Mastered Songs, NEW SONGS, New Tracks

In [ ]:
music_out = DESKTOP / 'MUSIC OUTPUT'
print('=== MUSIC OUTPUT ===')

# Group files by name (without extension) to find duplicates
name_map = defaultdict(list)
audio_ext = {'.mp3', '.wav', '.flac', '.aiff', '.aif', '.m4a', '.ogg'}

for f in music_out.rglob('*'):
    if f.is_file() and f.suffix.lower() in audio_ext:
        key = f.stem.upper().strip()
        name_map[key].append(f)

dups = {k: v for k, v in name_map.items() if len(v) > 1}
print(f'Audio files with duplicate names: {len(dups)}')
for name, paths in sorted(dups.items()):
    print(f'\n  {name}')
    for p in paths:
        try:
            size = p.stat().st_size / (1024*1024)
            mtime = datetime.fromtimestamp(p.stat().st_mtime).strftime('%Y-%m-%d')
        except:
            size, mtime = 0, 'unknown'
        rel = p.relative_to(music_out)
        print(f'    {str(rel):<60} {size:.1f} MB  {mtime}')

if not dups:
    print('No duplicates found — MUSIC OUTPUT is clean.')


## 8. Audit — Loose Files on Desktop Root
Anything sitting directly on the desktop (not in a folder) that shouldn't be there.

In [ ]:
loose = [f for f in DESKTOP.iterdir() if f.is_file()]
print(f'=== LOOSE FILES ON DESKTOP ROOT ({len(loose)}) ===')
for f in sorted(loose, key=lambda x: x.stat().st_mtime, reverse=True):
    try:
        size = f.stat().st_size / 1024
        mtime = datetime.fromtimestamp(f.stat().st_mtime).strftime('%Y-%m-%d')
    except:
        size, mtime = 0, 'unknown'
    print(f'  {f.name:<60} {size:>8.1f} KB  {mtime}')

# Suggest destinations for common types
print('\n--- Suggested moves ---')
ext_map = {
    '.lnk': 'Apps (find right subcategory)',
    '.url': 'URLS',
    '.pdf': 'ADMINISTRATION or MUSIC ADMIN',
    '.xlsx': 'ADMINISTRATION or FINANCES',
    '.docx': 'ADMINISTRATION or MUSIC ADMIN',
    '.mp3': 'MUSIC OUTPUT',
    '.wav': 'MUSIC OUTPUT',
    '.png': 'MUSIC MARKETING or AUDIO VISUAL',
    '.jpg': 'MUSIC MARKETING or AUDIO VISUAL',
}
for f in loose:
    suggestion = ext_map.get(f.suffix.lower(), 'GENERAL ITEMS or MISCEL')
    print(f'  {f.name:<50} -> {suggestion}')


## 9. Final Report & Save Log

In [ ]:
print('=== CLEANUP SUMMARY ===')
print(f'Mode: {"DRY RUN (nothing changed)" if DRY_RUN else "LIVE (changes applied)"}')
print(f'Actions logged: {len(log_lines)}')
print()
for line in log_lines:
    print(line)

# Save log to disk
timestamp = datetime.now().strftime('%Y-%m-%d_%H%M')
log_file = Path(str(LOG_PATH).replace('.txt', f'_{timestamp}.txt'))
log_file.write_text('\n'.join(log_lines), encoding='utf-8')
print(f'\nLog saved to: {log_file}')
